In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import f_oneway
from scipy.stats import tukey_hsd

In [ ]:
df = pd.read_csv("/Users/sametcan/Desktop/Thesis/Test_Batch/Data/Thesis_Study_Data_180.csv")
df = df[df['StartDate'] >= '2026-04-24'].reset_index(drop=True)
df

In [ ]:
# Remove the first two Qualtrics rows
df = df.iloc[2:].reset_index(drop=True)
df

In [ ]:
df_raw = df.copy()

# Check unfinished
print("Unfinished:", (df_raw['Finished'] == 'False').sum())

# Check attention check failures
print("Failed attention check:", (df_raw['Attention Check'] == 'Strongly Disagree').sum())

# Apply both filters
df = df_raw[(df_raw['Finished'] == 'True') & (df_raw['Attention Check'] == 'Strongly Agree')].reset_index(drop=True)
print(f"Remaining participants: {len(df)}")

In [ ]:
# Maps each ranking column prefix → (nudge, list_type)
GENDER_CONDITION_MAP = {
    'Gender Unbiased NN': ('Control',    'Unbiased'),
    'Gender Malebias NN': ('Control',    'Male-biased'),
    'Gender Fembias NN':  ('Control',    'Female-biased'),
    'Gender Unbiased AN': ('Automatic',  'Unbiased'),
    'Gender Malebias AN': ('Automatic',  'Male-biased'),
    'Gender Fembias AN':  ('Automatic',  'Female-biased'),
    'Gender Unbiased RN': ('Reflective', 'Unbiased'),
    'Gender Malebias RN': ('Reflective', 'Male-biased'),
    'Gender Fembias RN':  ('Reflective', 'Female-biased'),
}

AUTO_CONDITION_MAP = {
    'Auto Unbiased NN':  ('Control',    'Unbiased'),
    'Auto biased NN':    ('Control',    'Biased'),
    'Auto Unbiased AN':  ('Automatic',  'Unbiased'),
    'Auto biased AN':    ('Automatic',  'Biased'),
    'Auto unbiased RN':  ('Reflective', 'Unbiased'),
    'Auto biased RN':    ('Reflective', 'Biased'),
}

def get_gender_condition(row):
    for prefix, (nudge, list_type) in GENDER_CONDITION_MAP.items():
        col = prefix + ' 1'
        if col in df.columns and pd.notna(row[col]) and str(row[col]).strip() not in ('', 'nan'):
            return nudge, list_type
    return None, None

def get_auto_condition(row):
    for prefix, (nudge, list_type) in AUTO_CONDITION_MAP.items():
        col = prefix + ' 1'
        if col in df.columns and pd.notna(row[col]) and str(row[col]).strip() not in ('', 'nan'):
            return nudge, list_type
    return None, None

df[['gender_nudge', 'gender_list']] = df.apply(get_gender_condition, axis=1, result_type='expand')
df[['auto_nudge', 'auto_list']]     = df.apply(get_auto_condition,   axis=1, result_type='expand')

print("Gender conditions assigned:", df['gender_nudge'].value_counts().to_dict())
print("Auto conditions assigned:",   df['auto_nudge'].value_counts().to_dict())

In [ ]:
total = len(df)

print(f"Total participants: {total}\n")

print("=== GENDER CONDITIONS ===")
gender_nudge = df.groupby('gender_nudge').size()
gender_list = df.groupby('gender_list').size()
gender_both = df.groupby(['gender_nudge', 'gender_list']).size()

print("\nBy nudge:")
for k, v in gender_nudge.items():
    print(f"  {k}: {v} ({v/total*100:.1f}%)")

print("\nBy list type:")
for k, v in gender_list.items():
    print(f"  {k}: {v} ({v/total*100:.1f}%)")

print("\nBy nudge x list:")
for k, v in gender_both.items():
    print(f"  {k[0]} x {k[1]}: {v} ({v/total*100:.1f}%)")

print("\n=== AUTOMATION CONDITIONS ===")
auto_nudge = df.groupby('auto_nudge').size()
auto_list = df.groupby('auto_list').size()
auto_both = df.groupby(['auto_nudge', 'auto_list']).size()

print("\nBy nudge:")
for k, v in auto_nudge.items():
    print(f"  {k}: {v} ({v/total*100:.1f}%)")

print("\nBy list type:")
for k, v in auto_list.items():
    print(f"  {k}: {v} ({v/total*100:.1f}%)")

print("\nBy nudge x list:")
for k, v in auto_both.items():
    print(f"  {k[0]} x {k[1]}: {v} ({v/total*100:.1f}%)")

In [ ]:
for idx, val in df[df['Feedback\xa0'].notna() & (df['Feedback\xa0'].str.strip() != '')]['Feedback\xa0'].items():
    print(f"Participant {idx}: {val}\n")

## --- Demographic Breakdown ---

In [ ]:
# ── DEMOGRAPHIC BREAKDOWN ────────────────────────────────────────────────────

# Strip trailing spaces from column names (handles 'Age ')
df.columns = df.columns.str.strip()

# ── 1. OVERALL DESCRIPTIVES ──────────────────────────────────────────────────

print("=" * 55)
print(f"FINAL SAMPLE: N = {len(df)}")
print("=" * 55)

# Age (categorical ranges in your data)
print("\n--- AGE ---")
age_counts = df['Age'].value_counts().sort_index()
for val, n in age_counts.items():
    print(f"  {val}: {n} ({n/len(df)*100:.1f}%)")

# Gender, Education, Sector
for col in ['Gender', 'Education', 'Sector']:
    print(f"\n--- {col.upper()} ---")
    counts = df[col].value_counts()
    for val, n in counts.items():
        print(f"  {val}: {n} ({n/len(df)*100:.1f}%)")

In [ ]:
# ── 2. BREAKDOWN BY CONDITION ────────────────────────────────────────────────

print("\n" + "=" * 55)
print("DEMOGRAPHICS BY GENDER BIAS NUDGE CONDITION")
print("=" * 55)

for col in ['Age', 'Gender', 'Education', 'Sector']:
    print(f"\n--- {col} × gender_nudge ---")
    ct     = pd.crosstab(df[col], df['gender_nudge'], margins=True)
    ct_pct = pd.crosstab(df[col], df['gender_nudge'], normalize='columns').mul(100).round(1)
    print(ct)
    print("\n% within condition:")
    print(ct_pct)

print("\n" + "=" * 55)
print("DEMOGRAPHICS BY AUTOMATION BIAS NUDGE CONDITION")
print("=" * 55)

for col in ['Age', 'Gender', 'Education', 'Sector']:
    print(f"\n--- {col} × auto_nudge ---")
    ct     = pd.crosstab(df[col], df['auto_nudge'], margins=True)
    ct_pct = pd.crosstab(df[col], df['auto_nudge'], normalize='columns').mul(100).round(1)
    print(ct)
    print("\n% within condition:")
    print(ct_pct)

In [ ]:
# ── 3. RANDOMIZATION CHECKS (CHI-SQUARE) ────────────────────────────────────

print("\n" + "=" * 55)
print("RANDOMIZATION CHECKS")
print("=" * 55)

print("\n[Gender Bias conditions]")
for col in ['Age', 'Gender', 'Education', 'Sector']:
    ct = pd.crosstab(df[col], df['gender_nudge'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    sig = "✓ ns" if p > .05 else "⚠ SIGNIFICANT"
    print(f"  {col}: χ²({dof}) = {chi2:.3f}, p = {p:.3f}  {sig}")

print("\n[Automation Bias conditions]")
for col in ['Age', 'Gender', 'Education', 'Sector']:
    ct = pd.crosstab(df[col], df['auto_nudge'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    sig = "✓ ns" if p > .05 else "⚠ SIGNIFICANT"
    print(f"  {col}: χ²({dof}) = {chi2:.3f}, p = {p:.3f}  {sig}")

# ── 4. CLEAN SUMMARY TABLE ──────────────────────────────

print("\n" + "=" * 55)
print("TABLE 1 — SAMPLE CHARACTERISTICS")
print("=" * 55)

for col in ['Age', 'Gender', 'Education', 'Sector']:
    print(f"\n{col}:")
    summary = pd.crosstab(df[col], df['gender_nudge'],
                          margins=True, margins_name='Total')
    print(summary)

In [ ]:
#Chi-square tests confirmed that participants were evenly distributed across conditions with respect to 
#age, gender, education, and sector (all ps > .15), indicating successful randomization.

In [ ]:
#The final sample consisted of 174 participants (85 female, 87 male). 
#The majority held a Bachelor's degree (42.5%) and were between 25–34 years of age (38.5%). 
#Participants were distributed across various professional sectors, with Technology/IT (28.2%) and Other (28.7%) being most represented. 
#Chi-square tests confirmed no significant demographic differences across conditions (all ps > .15).

## --- BIAS SCORE CALCULATION ---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BIAS SCORE CALCULATION — MDG@3 (Mean Discounted Gain at cutoff 3)
# ══════════════════════════════════════════════════════════════════════════════
#
# Each participant gets FOUR bias scores:
#
#   gender_mdg3_vs_objective
#       How well the participant's top-3 matches the objectively correct
#       ranking (by qualification score). Low score = participant made poor
#       selections regardless of what the list showed them.
#
#   gender_mdg3_vs_list
#       How closely the participant's top-3 follows the order of the list
#       they were shown (unbiased, male-biased, or female-biased).
#       High score = participant followed the list order closely.
#
#   auto_mdg3_vs_objective
#       Same as above but for the automation bias task. How well participant
#       selections match the objectively correct ranking.
#
#   auto_mdg3_vs_list
#       How closely participant selections follow the AI ranking shown to them
#       (accurate or inaccurate AI list). High score = participant followed
#       the AI recommendation closely.
#
# The key derived measure is the bias_delta:
#   bias_delta = mdg3_vs_list - mdg3_vs_objective
#   Positive value → participant was pulled toward the list, away from truth.
#   This is the direct measure of list-induced bias.
#
# Ties in objective ranking are handled with averaged rank relevance:
#   e.g. Ethan C. and Hannah R. are both score 13 → both get relevance 1.5
#   (average of ranks 1 and 2), so swapping them does not change the score.
#
# ── COLUMN NAME MAP ──────────────────────────────────────────────────────────

GENDER_RANK_COLS = {
    ('Control',    'Unbiased'):       ['Gender Unbiased NN 1', 'Gender Unbiased NN 2', 'Gender Unbiased NN 3'],
    ('Control',    'Male-biased'):    ['Gender Malebias NN 1', 'Gender Malebias NN 2', 'Gender Malebias NN 1.1'],  # 1.1 = rank 3
    ('Control',    'Female-biased'):  ['Gender Fembias NN 1',  'Gender Fembias NN 2',  'Gender Fembias NN 3'],
    ('Automatic',  'Unbiased'):       ['Gender Unbiased AN 1', 'Gender Unbiased AN 2', 'Gender Unbiased AN 3'],
    ('Automatic',  'Male-biased'):    ['Gender Malebias AN 1', 'Gender Malebias AN 2', 'Gender Malebias AN 3'],
    ('Automatic',  'Female-biased'):  ['Gender Fembias AN 1',  'Gender Fembias AN 2',  'Gender Fembias AN 3'],
    ('Reflective', 'Unbiased'):       ['Gender Unbiased RN 1', 'Gender Unbiased RN 2', 'Gender Unbiased RN 3'],
    ('Reflective', 'Male-biased'):    ['Gender Malebias RN 1', 'Gender Malebias RN 2', 'Gender Malebias RN 3'],
    ('Reflective', 'Female-biased'):  ['Gender Fembias RN 1',  'Gender Fembias RN 2',  'Gender Fembias RN 3'],
}

AUTO_RANK_COLS = {
    ('Control',    'Unbiased'): ['Auto Unbiased NN 1', 'Auto Unbiased NN 2', 'Auto Unbiased NN 3'],
    ('Control',    'Biased'):   ['Auto biased NN 1',   'Auto biased NN 2',   'Auto biased NN 3'],
    ('Automatic',  'Unbiased'): ['Auto Unbiased AN 1', 'Auto Unbiased AN 2', 'Auto Unbiased AN 3'],
    ('Automatic',  'Biased'):   ['Auto biased AN 1',   'Auto biased AN 2',   'Auto biased AN 3'],
    ('Reflective', 'Unbiased'): ['Auto unbiased RN 1', 'Auto unbiased RN 2', 'Auto unbiased RN 3'],
    ('Reflective', 'Biased'):   ['Auto biased RN 1',   'Auto biased RN 2',   'Auto biased RN 3'],
}

# ── ANONYMOUS → REAL NAME LOOKUPS ────────────────────────────────────────────

GENDER_AN_UNBIASED = {
    'Candidate 1': 'Hannah R.',  'Candidate 2': 'Ethan C.',
    'Candidate 3': 'Michael T.', 'Candidate 4': 'Jacob W.',
    'Candidate 5': 'Emily C.',   'Candidate 6': 'Jessica B.',
    'Candidate 7': 'Olivia P.',  'Candidate 8': 'Ashley M.',
    'Candidate 9': 'Ryan M.',    'Candidate 10': 'Daniel H.',
}

GENDER_AN_MALEBIAS = {
    'Candidate 1': 'Ethan C.',   'Candidate 2': 'Michael T.',
    'Candidate 3': 'Jacob W.',   'Candidate 4': 'Hannah R.',
    'Candidate 5': 'Ryan M.',    'Candidate 6': 'Daniel H.',
    'Candidate 7': 'Emily C.',   'Candidate 8': 'Jessica B.',
    'Candidate 9': 'Olivia P.',  'Candidate 10': 'Ashley M.',
}

GENDER_AN_FEMBIAS = {
    'Candidate 1': 'Hannah R.',  'Candidate 2': 'Emily C.',
    'Candidate 3': 'Jessica B.', 'Candidate 4': 'Ethan C.',
    'Candidate 5': 'Olivia P.',  'Candidate 6': 'Ashley M.',
    'Candidate 7': 'Michael T.', 'Candidate 8': 'Jacob W.',
    'Candidate 9': 'Ryan M.',    'Candidate 10': 'Daniel H.',
}

AUTO_UNBIASED_MAP = {
    'Candidate 1': 'Hannah R.',  'Candidate 2': 'Ethan C.',
    'Candidate 3': 'Michael T.', 'Candidate 4': 'Emily C.',
    'Candidate 5': 'Jacob W.',   'Candidate 6': 'Jessica B.',
    'Candidate 7': 'Olivia P.',  'Candidate 8': 'Ashley M.',
    'Candidate 9': 'Ryan M.',    'Candidate 10': 'Daniel H.',
}

AUTO_BIASED_MAP = {
    'Candidate 1': 'Hannah R.',  'Candidate 2': 'Emily C.',
    'Candidate 3': 'Jacob W.',   'Candidate 4': 'Michael T.',
    'Candidate 5': 'Ethan C.',   'Candidate 6': 'Jessica B.',
    'Candidate 7': 'Olivia P.',  'Candidate 8': 'Ashley M.',
    'Candidate 9': 'Ryan M.',    'Candidate 10': 'Daniel H.',
}

# ── OBJECTIVE RELEVANCE WEIGHTS (averaged rank for ties) ─────────────────────
# Source: qualification scores from data list creation notebook.
# Ties resolved by averaging the ranks they occupy.
#
#   Score 13 → Ethan C., Hannah R.     → tied ranks 1,2   → relevance 1.5
#   Score 11 → Michael T.              → rank 3            → relevance 3.0
#   Score 10 → Emily C., Jacob W.      → tied ranks 4,5   → relevance 4.5
#   Score  9 → Jessica B.              → rank 6            → relevance 6.0
#   Score  8 → Olivia P., Ryan M.,
#              Ashley M.               → tied ranks 7,8,9 → relevance 8.0
#   Score  7 → Daniel H.              → rank 10           → relevance 10.0

OBJECTIVE_RELEVANCE = {
    'Ethan C.':   1.5,
    'Hannah R.':  1.5,
    'Michael T.': 3.0,
    'Emily C.':   4.5,
    'Jacob W.':   4.5,
    'Jessica B.': 6.0,
    'Olivia P.':  8.0,
    'Ryan M.':    8.0,
    'Ashley M.':  8.0,
    'Daniel H.':  10.0,
}

# ── LIST-ORDER RELEVANCE WEIGHTS ──────────────────────────────────────────────
# Relevance = position in the list shown to the participant (1 = top of list).
# No ties here — each list has a fixed deterministic order.
#
# Gender lists (position the candidate appears in the list shown):
GENDER_LIST_RELEVANCE = {
    'Unbiased': {
        'Ethan C.': 1, 'Hannah R.': 2, 'Michael T.': 3, 'Emily C.': 4,
        'Jacob W.': 5, 'Jessica B.': 6, 'Olivia P.': 7, 'Ryan M.': 8,
        'Ashley M.': 9, 'Daniel H.': 10,
    },
    'Male-biased': {
        'Ethan C.': 1, 'Michael T.': 2, 'Jacob W.': 3, 'Hannah R.': 4,
        'Ryan M.': 5, 'Daniel H.': 6, 'Emily C.': 7, 'Jessica B.': 8,
        'Olivia P.': 9, 'Ashley M.': 10,
    },
    'Female-biased': {
        'Hannah R.': 1, 'Emily C.': 2, 'Jessica B.': 3, 'Ethan C.': 4,
        'Olivia P.': 5, 'Ashley M.': 6, 'Michael T.': 7, 'Jacob W.': 8,
        'Ryan M.': 9, 'Daniel H.': 10,
    },
}

# Automation bias lists (position in the AI ranking shown):
# Accurate AI: correct order (Hannah 1st, Ethan 2nd, Michael 3rd ...)
# Inaccurate AI: wrong order — Hannah stays 1st, but Emily/Jacob pushed up
AUTO_LIST_RELEVANCE = {
    'Unbiased': {
        'Hannah R.': 1, 'Ethan C.': 2, 'Michael T.': 3, 'Emily C.': 4,
        'Jacob W.': 5, 'Jessica B.': 6, 'Olivia P.': 7, 'Ashley M.': 8,
        'Ryan M.': 9, 'Daniel H.': 10,
    },
    'Biased': {
        'Hannah R.': 1, 'Emily C.': 2, 'Jacob W.': 3, 'Michael T.': 4,
        'Ethan C.': 5, 'Jessica B.': 6, 'Olivia P.': 7, 'Ashley M.': 8,
        'Ryan M.': 9, 'Daniel H.': 10,
    },
}

# ── MDG@3 CORE FUNCTION ───────────────────────────────────────────────────────

def mdg3(selected_names, relevance_dict):
    """
    Compute Mean Discounted Gain at cutoff 3.

    selected_names : list of 3 real candidate names in the order the
                     participant ranked them (rank 1 first).
    relevance_dict : dict mapping candidate name → relevance weight.
                     For vs-objective: lower number = better (rank 1.5 is top).
                     For vs-list: lower number = better (position 1 is top).

    Because lower relevance = better candidate, we invert by using
    (max_relevance + 1 - rel) so that the best candidate gets the highest
    gain value in the discounted sum.

    Returns a score in [0, 1] (normalized by the ideal MDG@3).
    """
    max_rel = max(relevance_dict.values())

    def gain(name):
        rel = relevance_dict.get(name, max_rel + 1)  # unknown → worst possible
        return max_rel + 1 - rel                      # invert: best → highest gain

    # Discounted gain for participant's ranking
    score = sum(
        gain(name) / np.log2(rank + 1)
        for rank, name in enumerate(selected_names[:3], start=1)
    )

    # Ideal score: best 3 candidates in perfect order
    best3 = sorted(relevance_dict.items(), key=lambda x: x[1])[:3]
    ideal = sum(
        gain(name) / np.log2(rank + 1)
        for rank, (name, _) in enumerate(best3, start=1)
    )

    return round(score / ideal, 4) if ideal > 0 else np.nan


# ── HELPER ────────────────────────────────────────────────────────────────────

def get_top3_values(row, cols):
    vals = []
    for c in cols:
        v = row.get(c, None)
        if pd.notna(v) and str(v).strip() not in ('', 'nan'):
            vals.append(str(v).strip())
    return vals


# ── GENDER BIAS SCORES ────────────────────────────────────────────────────────

def gender_bias_scores(row):
    """
    Returns (mdg3_vs_objective, mdg3_vs_list) for the gender bias task.

    mdg3_vs_objective : Did the participant pick the objectively best
                        candidates, ignoring what the list suggested?
    mdg3_vs_list      : Did the participant follow the order of the list
                        they were shown?
    """
    nudge = row.get('gender_nudge')
    glist = row.get('gender_list')
    key   = (nudge, glist)

    if key not in GENDER_RANK_COLS:
        return np.nan, np.nan

    cols = GENDER_RANK_COLS[key]
    top3 = get_top3_values(row, cols)
    if len(top3) < 3:
        return np.nan, np.nan

    # Automatic nudge uses anonymous Candidate IDs → resolve to real names
    if nudge == 'Automatic':
        if glist == 'Unbiased':      lookup = GENDER_AN_UNBIASED
        elif glist == 'Male-biased': lookup = GENDER_AN_MALEBIAS
        else:                        lookup = GENDER_AN_FEMBIAS
        top3 = [lookup.get(n, n) for n in top3]

    mdg3_obj  = mdg3(top3, OBJECTIVE_RELEVANCE)
    mdg3_list = mdg3(top3, GENDER_LIST_RELEVANCE[glist])

    return mdg3_obj, mdg3_list


# ── AUTOMATION BIAS SCORES ────────────────────────────────────────────────────

def auto_bias_scores(row):
    """
    Returns (mdg3_vs_objective, mdg3_vs_list) for the automation bias task.

    mdg3_vs_objective : Did the participant pick the objectively best
                        candidates, regardless of the AI ranking shown?
    mdg3_vs_list      : Did the participant follow the AI ranking they
                        were shown (accurate or inaccurate)?
    """
    nudge = row.get('auto_nudge')
    alist = row.get('auto_list')
    key   = (nudge, alist)

    if key not in AUTO_RANK_COLS:
        return np.nan, np.nan

    cols = AUTO_RANK_COLS[key]
    top3 = get_top3_values(row, cols)
    if len(top3) < 3:
        return np.nan, np.nan

    lookup = AUTO_UNBIASED_MAP if alist == 'Unbiased' else AUTO_BIASED_MAP
    top3   = [lookup.get(n, n) for n in top3]

    mdg3_obj  = mdg3(top3, OBJECTIVE_RELEVANCE)
    mdg3_list = mdg3(top3, AUTO_LIST_RELEVANCE[alist])

    return mdg3_obj, mdg3_list


# ── APPLY TO DATAFRAME ────────────────────────────────────────────────────────

df[['gender_mdg3_vs_objective', 'gender_mdg3_vs_list']] = df.apply(
    lambda row: pd.Series(gender_bias_scores(row)), axis=1
)

df[['auto_mdg3_vs_objective', 'auto_mdg3_vs_list']] = df.apply(
    lambda row: pd.Series(auto_bias_scores(row)), axis=1
)

# Bias delta: positive = participant followed the list more than objective truth
# Only meaningful for biased lists (Male-biased, Female-biased, Auto Biased)
df['gender_bias_delta'] = df['gender_mdg3_vs_list'] - df['gender_mdg3_vs_objective']
df['auto_bias_delta']   = df['auto_mdg3_vs_list']   - df['auto_mdg3_vs_objective']

# ── SANITY CHECK ──────────────────────────────────────────────────────────────

print("=== SANITY CHECK — Gender (first 5 non-null rows) ===")
g_sample = df[df['gender_mdg3_vs_objective'].notna()].head(5)
for _, row in g_sample.iterrows():
    nudge = row['gender_nudge']; glist = row['gender_list']
    cols  = GENDER_RANK_COLS.get((nudge, glist), [])
    picks = get_top3_values(row, cols)
    print(
        f"  {nudge:10} | {glist:14} | picks: {picks}"
        f"\n             → vs_objective: {row['gender_mdg3_vs_objective']:.3f}"
        f"  vs_list: {row['gender_mdg3_vs_list']:.3f}"
        f"  delta: {row['gender_bias_delta']:.3f}\n"
    )

print("=== SANITY CHECK — Automation (first 5 non-null rows) ===")
a_sample = df[df['auto_mdg3_vs_objective'].notna()].head(5)
for _, row in a_sample.iterrows():
    nudge = row['auto_nudge']; alist = row['auto_list']
    cols  = AUTO_RANK_COLS.get((nudge, alist), [])
    picks = get_top3_values(row, cols)
    print(
        f"  {nudge:10} | {alist:10} | picks: {picks}"
        f"\n             → vs_objective: {row['auto_mdg3_vs_objective']:.3f}"
        f"  vs_list: {row['auto_mdg3_vs_list']:.3f}"
        f"  delta: {row['auto_bias_delta']:.3f}\n"
    )

# ── SUMMARY ───────────────────────────────────────────────────────────────────

print("\n" + "=" * 65)
print("BIAS SCORES — OVERVIEW")
print("=" * 65)

for label, obj_col, list_col, delta_col, nudge_col, list_col_name in [
    ("GENDER BIAS",     'gender_mdg3_vs_objective', 'gender_mdg3_vs_list',
     'gender_bias_delta', 'gender_nudge', 'gender_list'),
    ("AUTOMATION BIAS", 'auto_mdg3_vs_objective',   'auto_mdg3_vs_list',
     'auto_bias_delta',   'auto_nudge',   'auto_list'),
]:
    print(f"\n{'─'*65}")
    print(f"  {label}")
    print(f"{'─'*65}")

    for col, desc in [
        (obj_col,   "MDG@3 vs Objective  (higher = closer to true best candidates)"),
        (list_col,  "MDG@3 vs List       (higher = more closely followed the list)"),
        (delta_col, "Bias Delta          (positive = list pulled participant away from truth)"),
    ]:
        print(f"\n  {desc}")
        print(f"    Mean:    {df[col].mean():.3f}")
        print(f"    Std:     {df[col].std():.3f}")
        print(f"    Median:  {df[col].median():.3f}")
        print(f"    Missing: {df[col].isna().sum()}")

    print(f"\n  By nudge condition (bias delta):")
    print(df.groupby(nudge_col)[delta_col].agg(['mean', 'std', 'count']).round(3))

    print(f"\n  By list type (bias delta):")
    print(df.groupby(list_col_name)[delta_col].agg(['mean', 'std', 'count']).round(3))

    print(f"\n  By nudge × list (bias delta):")
    print(df.groupby([nudge_col, list_col_name])[delta_col]
            .agg(['mean', 'std', 'count']).round(3))

## --- Questionnaire Analysis ---

In [ ]:
# PERCEPTION SCALE CONSTRUCTION

# Word → number map for Autonomy-style columns
likert_word_map = {
    'Strongly disagree': 1,
    'Disagree': 2,
    'Somewhat disagree': 3,
    'Neither agree nor disagree': 4,
    'Somewhat agree': 5,
    'Agree': 6,
    'Strongly agree': 7,
}

def parse_likert(val):
    """Handle both '4 = Neither agree nor disagree' and 'Agree' formats."""
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    # Format: "4 = Some label" → extract the leading digit
    if '=' in val:
        try:
            return float(val.split('=')[0].strip())
        except:
            return np.nan
    # Format: plain word
    return float(likert_word_map.get(val, np.nan))

# Apply to all perception columns
perception_cols = {
    'Perceived Fairness_1': 'fair', 'Perceived Fairness_2': 'fair',
    'Perceived Fairness_3': 'fair', 'Perceived Fairness_4': 'fair',
    'Perceived Transparen_1': 'trans', 'Perceived Transparen_2': 'trans',
    'Perceived Transparen_3': 'trans', 'Perceived Transparen_4': 'trans',
    'Perceived Autonomy_1': 'auto', 'Perceived Autonomy_2': 'auto',
    'Perceived Autonomy_3': 'auto',
}

for col in perception_cols:
    df[col] = df[col].apply(parse_likert)

# Composite scores
df['fairness_score']     = df[['Perceived Fairness_1', 'Perceived Fairness_2', 'Perceived Fairness_3', 'Perceived Fairness_4']].mean(axis=1)
df['transparency_score'] = df[['Perceived Transparen_1', 'Perceived Transparen_2', 'Perceived Transparen_3', 'Perceived Transparen_4']].mean(axis=1)
df['autonomy_score']     = df[['Perceived Autonomy_1', 'Perceived Autonomy_2', 'Perceived Autonomy_3']].mean(axis=1)

# Sanity check
print("=== PERCEPTION SCORES — OVERVIEW ===\n")
for name, col in [('Fairness', 'fairness_score'), ('Transparency', 'transparency_score'), ('Autonomy', 'autonomy_score')]:
    print(f"{name}:")
    print(f"  Mean: {df[col].mean():.3f}  |  Std: {df[col].std():.3f}  |  Missing: {df[col].isna().sum()}")
    print(f"  Min: {df[col].min():.1f}  |  Max: {df[col].max():.1f}\n")

In [ ]:
# INTERNAL CONSISTENCY (Cronbach's Alpha)

def cronbach_alpha(df_items):
    df_items = df_items.dropna()
    k = df_items.shape[1]
    item_vars = df_items.var(axis=0, ddof=1)
    total_var = df_items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)

scales = {
    'Fairness':     ['Perceived Fairness_1', 'Perceived Fairness_2', 'Perceived Fairness_3', 'Perceived Fairness_4'],
    'Transparency': ['Perceived Transparen_1', 'Perceived Transparen_2', 'Perceived Transparen_3', 'Perceived Transparen_4'],
    'Autonomy':     ['Perceived Autonomy_1', 'Perceived Autonomy_2', 'Perceived Autonomy_3'],
}

print("=== CRONBACH'S ALPHA ===\n")
for scale, cols in scales.items():
    alpha = cronbach_alpha(df[cols])
    interp = "excellent" if alpha >= 0.9 else "good" if alpha >= 0.8 else "acceptable" if alpha >= 0.7 else "questionable"
    print(f"  {scale}: α = {alpha:.3f}  ({interp})")

## --- Normality & Homogeneity of Variance ---

In [ ]:
# ASSUMPTION CHECKS

from scipy.stats import shapiro, levene

# Outcomes: (dependent variable, nudge grouping col, list grouping col)
outcomes = {
    'gender_mdg3_vs_objective': ('gender_nudge', 'gender_list', 'gender_mdg3_vs_objective'),
    'gender_mdg3_vs_list':      ('gender_nudge', 'gender_list', 'gender_mdg3_vs_list'),
    'gender_bias_delta':        ('gender_nudge', 'gender_list', 'gender_bias_delta'),
    'auto_mdg3_vs_objective':   ('auto_nudge',   'auto_list',   'auto_mdg3_vs_objective'),
    'auto_mdg3_vs_list':        ('auto_nudge',   'auto_list',   'auto_mdg3_vs_list'),
    'auto_bias_delta':          ('auto_nudge',   'auto_list',   'auto_bias_delta'),
    'fairness_score':           ('gender_nudge', 'gender_list', 'fairness_score'),
    'transparency_score':       ('gender_nudge', 'gender_list', 'transparency_score'),
    'autonomy_score':           ('gender_nudge', 'gender_list', 'autonomy_score'),
}

print("=== NORMALITY (Shapiro-Wilk per nudge × list cell) ===")
print("p > .05 = normal\n")

for label, (nudge_col, list_col, score_col) in outcomes.items():
    if score_col not in df.columns:
        print(f"--- {label} --- SKIPPED (column not found)\n")
        continue
    print(f"--- {label} ---")
    data = df[df[score_col].notna()]
    for (nudge, lst), group in data.groupby([nudge_col, list_col]):
        vals = group[score_col].values
        if len(vals) < 3:
            print(f"  {nudge} | {lst}: too few observations (n={len(vals)})")
            continue
        stat, p = shapiro(vals)
        result = "✓ normal" if p > .05 else "⚠ non-normal"
        print(f"  {nudge:10} | {lst:14} (n={len(vals):2d}): W = {stat:.3f}, p = {p:.3f}  {result}")
    print()

print("\n=== HOMOGENEITY OF VARIANCE (Levene's across nudge conditions) ===")
print("p > .05 = variances are equal\n")

for label, (nudge_col, list_col, score_col) in outcomes.items():
    if score_col not in df.columns:
        continue
    data = df[df[score_col].notna()]
    groups = [g[score_col].values for _, g in data.groupby(nudge_col)]
    if len(groups) < 2:
        continue
    stat, p = levene(*groups)
    result = "✓ equal" if p > .05 else "⚠ unequal"
    print(f"  {label}: F = {stat:.3f}, p = {p:.3f}  {result}")

## --- Gender ANOVA ---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# H1a — Two-way ANOVA: gender_nudge × gender_list → gender bias scores
#
# Run twice:
#   (1) gender_mdg3_vs_objective — all list conditions (unbiased, male, female)
#       Did nudge or list type affect how objectively good participants' picks were?
#
#   (2) gender_mdg3_vs_list — biased list conditions only (male, female)
#       Did nudge or list type affect how closely participants followed the list?
#       Unbiased excluded here because vs_list ≈ vs_objective on an unbiased list,
#       making it analytically redundant.
# ══════════════════════════════════════════════════════════════════════════════

import pingouin as pg

def run_gender_anova(dv, data, label):
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"  DV: {dv}")
    print(f"  N = {len(data)}")
    print(f"{'='*70}")

    aov = pg.anova(data=data.dropna(subset=[dv]),
                   dv=dv,
                   between=['gender_nudge', 'gender_list'],
                   detailed=True)

    print("\n  ANOVA Table:")
    for _, row in aov.iterrows():
        sig = '✓ Significant' if row['p_unc'] < 0.05 else 'ns'
        print(f"    {row['Source']:<35} F = {row['F']:6.3f}, p = {row['p_unc']:.3f}, "
              f"η²p = {row['np2']:.3f}  {sig}")

    print("\n  Means by nudge condition:")
    for name, grp in data.groupby('gender_nudge'):
        vals = grp[dv].dropna()
        print(f"    {name:<15} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    print("\n  Means by list type:")
    for name, grp in data.groupby('gender_list'):
        vals = grp[dv].dropna()
        print(f"    {name:<20} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    print("\n  Means by nudge × list:")
    for (nudge, lst), grp in data.groupby(['gender_nudge', 'gender_list']):
        vals = grp[dv].dropna()
        print(f"    {nudge:<12} | {lst:<15} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    # Post-hoc Tukey for any significant main effects
    for factor in ['gender_nudge', 'gender_list']:
        p_factor = aov[aov['Source'] == factor]['p_unc'].values
        if len(p_factor) > 0 and p_factor[0] < 0.05:
            print(f"\n  Tukey post-hoc for {factor}:")
            posthoc = pg.pairwise_tukey(data=data.dropna(subset=[dv]),
                                        dv=dv, between=factor)
            for _, r in posthoc.iterrows():
                sig = '✓' if r['p_tukey'] < 0.05 else 'ns'
                print(f"    {r['A']} vs {r['B']}: diff = {r['diff']:.3f}, "
                      f"p = {r['p_tukey']:.3f}  {sig}")

    # Interaction post-hoc if significant
    p_interaction = aov[aov['Source'] == 'gender_nudge * gender_list']['p_unc'].values
    if len(p_interaction) > 0 and p_interaction[0] < 0.05:
        print(f"\n  Interaction significant — simple effects (nudge within each list type):")
        for lst_type, grp in data.groupby('gender_list'):
            print(f"\n    List = {lst_type}:")
            posthoc = pg.pairwise_tukey(data=grp.dropna(subset=[dv]),
                                        dv=dv, between='gender_nudge')
            for _, r in posthoc.iterrows():
                sig = '✓' if r['p_tukey'] < 0.05 else 'ns'
                print(f"      {r['A']} vs {r['B']}: diff = {r['diff']:.3f}, "
                      f"p = {r['p_tukey']:.3f}  {sig}")


# ── RUN 1: vs_objective — all three list conditions ───────────────────────────
run_gender_anova(
    dv='gender_mdg3_vs_objective',
    data=df.dropna(subset=['gender_mdg3_vs_objective']),
    label="H1a (i): nudge × list → MDG@3 vs Objective (all list conditions)"
)

# ── RUN 2: vs_list — biased conditions only ───────────────────────────────────
run_gender_anova(
    dv='gender_mdg3_vs_list',
    data=df[df['gender_list'].isin(['Male-biased', 'Female-biased'])].dropna(subset=['gender_mdg3_vs_list']),
    label="H1a (ii): nudge × list → MDG@3 vs List (biased conditions only: Male-biased & Female-biased)"
)

## --- Automation ANOVA ---

In [ ]:
# H1b — Two-way ANOVA: auto_nudge × auto_list → automation bias scores
#
# Run twice:
#   (1) auto_mdg3_vs_objective — all list conditions (unbiased, biased)
#       Did nudge or list type affect how objectively good participants' picks were?
#
#   (2) auto_mdg3_vs_list — all list conditions (unbiased, biased)
#       Did nudge or list type affect how closely participants followed
#       the AI ranking they were shown (accurate or inaccurate)?

def run_auto_anova(dv, data, label):
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"  DV: {dv}")
    print(f"  N = {len(data)}")
    print(f"{'='*70}")

    aov = pg.anova(data=data.dropna(subset=[dv]),
                   dv=dv,
                   between=['auto_nudge', 'auto_list'],
                   detailed=True)

    print("\n  ANOVA Table:")
    for _, row in aov.iterrows():
        sig = '✓ Significant' if row['p_unc'] < 0.05 else 'ns'
        print(f"    {row['Source']:<35} F = {row['F']:6.3f}, p = {row['p_unc']:.3f}, "
              f"η²p = {row['np2']:.3f}  {sig}")

    print("\n  Means by nudge condition:")
    for name, grp in data.groupby('auto_nudge'):
        vals = grp[dv].dropna()
        print(f"    {name:<15} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    print("\n  Means by list type:")
    for name, grp in data.groupby('auto_list'):
        vals = grp[dv].dropna()
        print(f"    {name:<15} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    print("\n  Means by nudge × list:")
    for (nudge, lst), grp in data.groupby(['auto_nudge', 'auto_list']):
        vals = grp[dv].dropna()
        print(f"    {nudge:<12} | {lst:<10} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    # Post-hoc Tukey for any significant main effects
    for factor in ['auto_nudge', 'auto_list']:
        p_factor = aov[aov['Source'] == factor]['p_unc'].values
        if len(p_factor) > 0 and p_factor[0] < 0.05:
            print(f"\n  Tukey post-hoc for {factor}:")
            posthoc = pg.pairwise_tukey(data=data.dropna(subset=[dv]),
                                        dv=dv, between=factor)
            for _, r in posthoc.iterrows():
                sig = '✓' if r['p_tukey'] < 0.05 else 'ns'
                print(f"    {r['A']} vs {r['B']}: diff = {r['diff']:.3f}, "
                      f"p = {r['p_tukey']:.3f}  {sig}")

    # Interaction post-hoc if significant
    p_interaction = aov[aov['Source'] == 'auto_nudge * auto_list']['p_unc'].values
    if len(p_interaction) > 0 and p_interaction[0] < 0.05:
        print(f"\n  Interaction significant — simple effects (nudge within each list type):")
        for lst_type, grp in data.groupby('auto_list'):
            print(f"\n    List = {lst_type}:")
            posthoc = pg.pairwise_tukey(data=grp.dropna(subset=[dv]),
                                        dv=dv, between='auto_nudge')
            for _, r in posthoc.iterrows():
                sig = '✓' if r['p_tukey'] < 0.05 else 'ns'
                print(f"      {r['A']} vs {r['B']}: diff = {r['diff']:.3f}, "
                      f"p = {r['p_tukey']:.3f}  {sig}")


# ── RUN 1: vs_objective — all list conditions ─────────────────────────────────
run_auto_anova(
    dv='auto_mdg3_vs_objective',
    data=df.dropna(subset=['auto_mdg3_vs_objective']),
    label="H1b (i): nudge × list → MDG@3 vs Objective (all list conditions)"
)

# ── RUN 2: vs_list — all list conditions ──────────────────────────────────────
run_auto_anova(
    dv='auto_mdg3_vs_list',
    data=df.dropna(subset=['auto_mdg3_vs_list']),
    label="H1b (ii): nudge × list → MDG@3 vs List (all list conditions)"
)

## --- H1C ----

In [ ]:
# Effect of nudge on gender bias (vs control)
gender_ctrl = df[df['gender_nudge'] == 'Control']['gender_mdg3_vs_objective'].dropna()
gender_refl = df[df['gender_nudge'] == 'Reflective']['gender_mdg3_vs_objective'].dropna()
gender_auto = df[df['gender_nudge'] == 'Automatic']['gender_mdg3_vs_objective'].dropna()

# Effect of nudge on automation bias (vs control)
auto_ctrl = df[df['auto_nudge'] == 'Control']['auto_mdg3_vs_objective'].dropna()
auto_refl = df[df['auto_nudge'] == 'Reflective']['auto_mdg3_vs_objective'].dropna()
auto_auto = df[df['auto_nudge'] == 'Automatic']['auto_mdg3_vs_objective'].dropna()

# Compare Cohen's d for each nudge type across bias types
for nudge_label, g_grp, a_grp in [
    ('Reflective', gender_refl, auto_refl),
    ('Automatic',  gender_auto, auto_auto),
]:
    d_gender = (g_grp.mean() - gender_ctrl.mean()) / np.sqrt((g_grp.std()**2 + gender_ctrl.std()**2) / 2)
    d_auto   = (a_grp.mean() - auto_ctrl.mean())   / np.sqrt((a_grp.std()**2 + auto_ctrl.std()**2) / 2)
    print(f"{nudge_label}: Cohen's d (gender) = {d_gender:.3f}, Cohen's d (automation) = {d_auto:.3f}")

## ---  Gender Questionnaire ANOVA ---

In [ ]:
# H2 — Two-way ANOVA: nudge × list → fairness, transparency, autonomy scores

def run_perceptual_anova(dv, nudge_col, list_col, label):
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"  DV: {dv}")
    data = df.dropna(subset=[dv])
    print(f"  N = {len(data)}")
    print(f"{'='*70}")

    aov = pg.anova(data=data, dv=dv,
                   between=[nudge_col, list_col],
                   detailed=True)

    print("\n  ANOVA Table:")
    for _, row in aov.iterrows():
        sig = '✓ Significant' if row['p_unc'] < 0.05 else 'ns'
        print(f"    {row['Source']:<35} F = {row['F']:6.3f}, p = {row['p_unc']:.3f}, "
              f"η²p = {row['np2']:.3f}  {sig}")

    print("\n  Means by nudge condition:")
    for name, grp in data.groupby(nudge_col):
        vals = grp[dv]
        print(f"    {name:<15} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    print("\n  Means by list type:")
    for name, grp in data.groupby(list_col):
        vals = grp[dv]
        print(f"    {name:<20} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    print("\n  Means by nudge × list:")
    for (nudge, lst), grp in data.groupby([nudge_col, list_col]):
        vals = grp[dv]
        print(f"    {nudge:<12} | {lst:<15} M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

    # Post-hoc Tukey for significant main effects
    for factor in [nudge_col, list_col]:
        p_factor = aov[aov['Source'] == factor]['p_unc'].values
        if len(p_factor) > 0 and p_factor[0] < 0.05:
            print(f"\n  Tukey post-hoc for {factor}:")
            posthoc = pg.pairwise_tukey(data=data, dv=dv, between=factor)
            for _, r in posthoc.iterrows():
                sig = '✓' if r['p_tukey'] < 0.05 else 'ns'
                print(f"    {r['A']} vs {r['B']}: diff = {r['diff']:.3f}, "
                      f"p = {r['p_tukey']:.3f}  {sig}")

    # Interaction post-hoc if significant
    interaction_source = f'{nudge_col} * {list_col}'
    p_interaction = aov[aov['Source'] == interaction_source]['p_unc'].values
    if len(p_interaction) > 0 and p_interaction[0] < 0.05:
        print(f"\n  Interaction significant — simple effects (nudge within each list type):")
        for lst_type, grp in data.groupby(list_col):
            print(f"\n    List = {lst_type}:")
            posthoc = pg.pairwise_tukey(data=grp, dv=dv, between=nudge_col)
            for _, r in posthoc.iterrows():
                sig = '✓' if r['p_tukey'] < 0.05 else 'ns'
                print(f"      {r['A']} vs {r['B']}: diff = {r['diff']:.3f}, "
                      f"p = {r['p_tukey']:.3f}  {sig}")


# ── Gender task perceptual outcomes ───────────────────────────────────────────
run_perceptual_anova('fairness_score',      'gender_nudge', 'gender_list',
                     'H2a (Gender): nudge × list → Fairness')
run_perceptual_anova('transparency_score',  'gender_nudge', 'gender_list',
                     'H2b (Gender): nudge × list → Transparency')
run_perceptual_anova('autonomy_score',      'gender_nudge', 'gender_list',
                     'H2c (Gender): nudge × list → Autonomy')

## ---  Automation Questionnaire ANOVA ---

In [ ]:
# ── Automation task perceptual outcomes ───────────────────────────────────────
run_perceptual_anova('fairness_score',      'auto_nudge', 'auto_list',
                     'H2a (Automation): nudge × list → Fairness')
run_perceptual_anova('transparency_score',  'auto_nudge', 'auto_list',
                     'H2b (Automation): nudge × list → Transparency')
run_perceptual_anova('autonomy_score',      'auto_nudge', 'auto_list',
                     'H2c (Automation): nudge × list → Autonomy')

## ---- H3 ----

In [ ]:
# H3a — Nudge effect on gender bias, separately for male and female participants
# Preregistered: independent samples t-test, nudge vs control, per participant gender
# DV: gender_mdg3_vs_objective

from scipy.stats import ttest_ind

print("=== H3a: Nudge effect by participant gender → gender_mdg3_vs_objective ===\n")

for sex in ['Male', 'Female']:
    print(f"  ── {sex} participants ──")
    control = df[(df['Gender'] == sex) & (df['gender_nudge'] == 'Control')]['gender_mdg3_vs_objective'].dropna()

    for nudge in ['Automatic', 'Reflective']:
        nudge_grp = df[(df['Gender'] == sex) & (df['gender_nudge'] == nudge)]['gender_mdg3_vs_objective'].dropna()

        t_stat, p_val = ttest_ind(nudge_grp, control)
        pooled_std = np.sqrt((nudge_grp.std()**2 + control.std()**2) / 2)
        cohens_d   = (nudge_grp.mean() - control.mean()) / pooled_std

        print(f"\n    {nudge} vs Control:")
        print(f"      {nudge}:  M = {nudge_grp.mean():.3f}, SD = {nudge_grp.std():.3f}, n = {len(nudge_grp)}")
        print(f"      Control: M = {control.mean():.3f}, SD = {control.std():.3f}, n = {len(control)}")
        print(f"      t = {t_stat:.3f}, p = {p_val:.3f}, d = {cohens_d:.3f}")
        print(f"      Significant: {'✓ Yes' if p_val < 0.05 else 'ns'} (α = 0.05)")
    print()

print("─" * 60)
print("  H3a supported if nudge effect is significant for males")
print("  but not for females (or significantly weaker for females).")

In [ ]:
# H3b — Independent samples t-test: AI experience → auto_mdg3_vs_objective
# AI-experienced: Technology/IT or Engineering
# Non-experienced: all other sectors (excluding 'Prefer not to share')
# Preregistered confirmatory analysis


from scipy.stats import ttest_ind

# Create grouping variable
tech_sectors = ['Technology / IT', 'Engineering']
prefer_not   = ['Prefer not to share this information']

df['ai_experience'] = df['Sector'].apply(
    lambda x: 'High' if x in tech_sectors
    else ('Exclude' if x in prefer_not else 'Low')
)

high_exp = df[df['ai_experience'] == 'High']['auto_mdg3_vs_objective'].dropna()
low_exp  = df[df['ai_experience'] == 'Low']['auto_mdg3_vs_objective'].dropna()

t_stat, p_val = ttest_ind(high_exp, low_exp)
pooled_std = np.sqrt((high_exp.std()**2 + low_exp.std()**2) / 2)
cohens_d   = (high_exp.mean() - low_exp.mean()) / pooled_std

print("=== H3b: AI Experience → auto_smdg3_vs_objective ===\n")
print(f"  High experience (Tech/IT + Engineering): M = {high_exp.mean():.3f}, SD = {high_exp.std():.3f}, n = {len(high_exp)}")
print(f"  Low experience (all other sectors):      M = {low_exp.mean():.3f}, SD = {low_exp.std():.3f}, n = {len(low_exp)}")
print(f"\n  t = {t_stat:.3f}, p = {p_val:.3f}, d = {cohens_d:.3f}")
print(f"  Significant: {'✓ Yes' if p_val < 0.05 else 'ns'} (α = 0.05)")
print(f"\n  Note: {df[df['ai_experience'] == 'Exclude'].shape[0]} participants excluded (Prefer not to share)")
print(f"\n  Sector breakdown:")
print(df[df['ai_experience'] != 'Exclude']['Sector'].value_counts())